# Predicting Employee Attrition
### Applying Label Encoding, One-Hot Encoding, and Ordinal Encoding

## Load and inspect the data

In [12]:
import pandas as pd

df = pd.read_csv('employee_attrition.csv')
df.head(10)

,EmployeeID,Name,Age,Gender,City,Department,Education,YearsExperience,MonthlyIncome,PerformanceRating,SatisfactionScore,Attrition
0,1000,Ava V.,33,Female,Bengaluru,Engineering,PhD,13,98014,Good,5,No
1,1001,Noah C.,28,Male,Delhi,Sales,High School,4,50392,Good,4,No
2,1002,Meera C.,27,Male,Hyderabad,Finance,Master,1,59889,Good,4,No
3,1003,Ava L.,44,Male,Pune,Support,Bachelor,21,97139,Good,5,No
4,1004,Sophia M.,24,Female,Pune,Support,Master,0,65642,Good,2,No
5,1005,Mason E.,28,Male,Hyderabad,Sales,Master,6,81326,Poor,3,No
6,1006,Emma C.,28,Female,Delhi,Support,Master,3,61861,Excellent,3,No
7,1007,Meera Z.,23,Female,Bengaluru,Engineering,Bachelor,0,47936,Good,3,No
8,1008,Isha G.,44,Female,Delhi,Sales,Bachelor,22,109785,Poor,3,Yes
9,1009,Mia K.,24,Female,Hyderabad,Engineering,Bachelor,0,47267,Excellent,4,No


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   EmployeeID         220 non-null    int64 
 1   Name               220 non-null    object
 2   Age                220 non-null    int64 
 3   Gender             220 non-null    object
 4   City               220 non-null    object
 5   Department         220 non-null    object
 6   Education          220 non-null    object
 7   YearsExperience    220 non-null    int64 
 8   MonthlyIncome      220 non-null    int64 
 9   PerformanceRating  220 non-null    object
 10  SatisfactionScore  220 non-null    int64 
 11  Attrition          220 non-null    object
dtypes: int64(5), object(7)
memory usage: 20.8+ KB


In [ ]:
df.describe(include='object')

## 2. study columns

| Column | Type | Notes |
|---|---|---|
| EmployeeID | identifier | drop before modeling |
| Name | identifier | drop before modeling |
| Age | numeric | already usable |
| Gender | categorical - **binary** | Male / Female |
| City | categorical - **nominal** | 6 cities, no order |
| Department | categorical - **nominal** | 6 departments, no order |
| Education | categorical - **ordinal** | High School < Bachelor < Master < PhD |
| YearsExperience | numeric | already usable |
| MonthlyIncome | numeric | already usable |
| PerformanceRating | categorical - **ordinal** | Poor < Average < Good < Excellent |
| SatisfactionScore | numeric (1-5 survey scale) | already usable |
| Attrition | categorical - **binary target** | Yes / No |


## 3. Clean the messy bits first

In [16]:
for col in ['Gender', 'City', 'Education', 'Department', 'PerformanceRating']:
    df[col] = df[col].str.strip()

df['Gender'].unique(), df['City'].unique(), df['Education'].unique()

(array(['Female', 'Male'], dtype=object),
 array(['Bengaluru', 'Delhi', 'Hyderabad', 'Pune', 'hyderabad', 'Mumbai',
        'Chennai'], dtype=object),
 array(['PhD', 'High School', 'Master', 'Bachelor', 'bachelor'],
       dtype=object))

## 4. Label Encoding — for binary columns and the target

In [17]:
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
df['Gender_encoded'] = le_gender.fit_transform(df['Gender'])

le_attrition = LabelEncoder()
df['Attrition_encoded'] = le_attrition.fit_transform(df['Attrition'])

print(dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))

print(dict(zip(le_attrition.classes_, le_attrition.transform(le_attrition.classes_))))


{'Female': np.int64(0), 'Male': np.int64(1)}
{'No': np.int64(0), 'Yes': np.int64(1)}


In [18]:
le_gender.classes_

array(['Female', 'Male'], dtype=object)

In [19]:
le_attrition.transform(le_attrition.classes_)

array([0, 1])

In [20]:
df[['Gender', 'Gender_encoded', 'Attrition', 'Attrition_encoded']].head()

,Gender,Gender_encoded,Attrition,Attrition_encoded
0,Female,0,No,0
1,Male,1,No,0
2,Male,1,No,0
3,Male,1,No,0
4,Female,0,No,0


## 5. One-Hot Encoding — for nominal columns

In [21]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ohe_array = ohe.fit_transform(df[['Department', 'City']])

ohe_df = pd.DataFrame(ohe_array, columns=ohe.get_feature_names_out(['Department', 'City']))
ohe_df.head()

,Department_Engineering,Department_Finance,Department_HR,Department_Marketing,Department_Sales,Department_Support,City_Bengaluru,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,City_Pune,City_hyderabad
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 6. Ordinal Encoding — for ranked categories


In [25]:
from sklearn.preprocessing import OrdinalEncoder

#education_order = ['High School', 'Bachelor', 'Master', 'Phd']
performance_order = ['Poor', 'Average', 'Good', 'Excellent']

ord_enc = OrdinalEncoder(categories=[ performance_order])
df[[ 'Performance_encoded']] = ord_enc.fit_transform(
    df[['PerformanceRating']]
)

df[['PerformanceRating', 'Performance_encoded']].head(8)

,PerformanceRating,Performance_encoded
0,Good,2.0
1,Good,2.0
2,Good,2.0
3,Good,2.0
4,Good,2.0
5,Poor,0.0
6,Excellent,3.0
7,Good,2.0


## 7. Assemble the final model-ready dataset

In [26]:
final_df = pd.concat([
    df[['Age', 'YearsExperience', 'MonthlyIncome', 'SatisfactionScore',
        'Gender_encoded', 'Performance_encoded']],
    ohe_df,
    df['Attrition_encoded'].rename('Attrition')
], axis=1)

final_df.head()

,Age,YearsExperience,MonthlyIncome,SatisfactionScore,Gender_encoded,Performance_encoded,Department_Engineering,Department_Finance,Department_HR,Department_Marketing,Department_Sales,Department_Support,City_Bengaluru,City_Chennai,City_Delhi,City_Hyderabad,City_Mumbai,City_Pune,City_hyderabad,Attrition
0,33,13,98014,5,0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,28,4,50392,4,1,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
2,27,1,59889,4,1,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,44,21,97139,5,1,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,24,0,65642,2,0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
